Notebook Author: Steven Camacho

This notebook runs an experiment to find and verify that Multilingual LLMs showcase informational asymmetry when prompted in different languages for the same event or topic, supporting that the idea that there is bias in multilingual NLP systems.

* IMPORTANT: If you don't already have a Hugging Face account or a Google Drive account, be sure to create both prior to running this notebook.

In [ ]:
# Step 1: Install dependencies for model prompting and entity extraction/counting

# Hugging Face for model calls and dataset
!pip install -q transformers accelerate sentencepiece huggingface_hub

# Spacy for entity extraction from model responses
!pip install -q -U spacy
!python3 -m spacy download en_core_web_trf
!python3 -m spacy download es_core_news_lg

# wikipedia
!pip install -q wikipedia-api

# NER Tracking
!pip install -q gliner

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 28.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 2.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   

In [ ]:
# for local saving of files
isTesting = False

In [ ]:
# Step 2: Mount Google Drive
if isTesting is False:
  from google.colab import drive
  drive.mount("/content/drive")

  PROJECT_DIR = "/content/drive/MyDrive/NLP_project"

In [ ]:
# Step 3: Imports and paths
import json
import gc
from pathlib import Path
import pandas as pd

import torch
import spacy
from transformers import pipeline
import wikipediaapi
from pprint import pprint
import re

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Paths
if isTesting:
    BASE_DIR = Path("/content/output")
else:
    BASE_DIR = Path("/content/drive/MyDrive/NLP_project/results")

# Dataset name controls the subfolder — change this when adding new datasets
DATASET_NAME = "natural_disasters"
RESULTS_DIR  = BASE_DIR / DATASET_NAME

# Subfolders
NER_DIR      = RESULTS_DIR / "ner"
GT_DIR       = RESULTS_DIR / "ground_truth"
ANALYSIS_DIR = RESULTS_DIR / "analysis"

for d in [RESULTS_DIR, NER_DIR, GT_DIR, ANALYSIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Fixed file paths
MODEL_OUTPUTS_PATH       = RESULTS_DIR  / "model_outputs.jsonl"
GT_ENTITIES_PATH         = GT_DIR       / "gt_entities.json"
GT_ENTITY_COUNTS_PATH    = GT_DIR       / "gt_entity_counts.csv"
MODEL_ENTITY_COUNTS_PATH = ANALYSIS_DIR / "model_entity_counts.csv"
ENTITY_COUNTS_PATH       = ANALYSIS_DIR / "entity_counts.csv"
TYPE_COUNTS_PATH         = ANALYSIS_DIR / "entity_type_counts.csv"
GT_RATIOS_PATH           = ANALYSIS_DIR / "gt_type_ratios.csv"

def ner_path(model_short, lang):
    """Returns the save path for NER entities for a given model/language."""
    return NER_DIR / f"{model_short}_{lang}.json"

print(f"Results will be saved to: {RESULTS_DIR}")

# load helpers
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Saved -> {path}")

def load_json(path):
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  Saved {len(records)} records -> {path}")

def load_jsonl(path):
    if Path(path).exists():
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))
        print(f"  Loaded {len(records)} records <- {path}")
        return records
    return None

def save_entities(entities_list, path):
    """Save a list-of-lists of (text, label) tuples."""
    serializable = [[[t, l] for t, l in event] for event in entities_list]
    save_json(serializable, path)

def load_entities(path):
    """Load and restore as list-of-lists of (text, label) tuples."""
    data = load_json(path)
    if data is None:
        return None
    print(f"  Loaded entities <- {path}")
    return [[(t, l) for t, l in event] for event in data]


GPU available: False
Results will be saved to: /content/output/natural_disasters


In [ ]:
# Step 4: declare datasets
natural_disasters = [
    "2004 Indian Ocean earthquake and tsunami",
    "1976 Tangshan earthquake",
    "1920 Haiyuan earthquake",
    "1923 Great Kantō earthquake",
    "1948 Ashgabat earthquake",
    "2011 Tōhoku earthquake and tsunami",
    "1995 Great Hanshin earthquake",
    "2023 Turkey–Syria earthquakes",
    "2008 Sichuan earthquake",
    "2011 Christchurch earthquake"
]

infrastructure_problems = [
    "2012 India blackouts",
    "2023 Pakistan blackout",
    "2022 Bangladesh blackout",
    "2021 Pakistan blackout",
    "2014 Bangladesh blackout",
    "2015 Pakistan blackout",
    "2019 Java blackout",
    "2005 Java–Bali blackout",
    "2015 Turkey blackout",
    "2009 Brazil and Paraguay blackout"
]

conflicts = [
    "War in Darfur",
    "Iraq War",
    "Mexican Drug War",
    "Central African Bush War",
    "2005 Bangladesh–India border clash",
    "Chadian Civil War",
    "Iraqi civil war",
    "Operation Astute",
    "2006 Lebanon War",
    "Somalia Civil War"
]

2004 Indian Ocean earthquake and tsunami


In [ ]:
# Step 5: Authenticate with Hugging Face for model calls
from huggingface_hub import login
login()  # paste your HF token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Step 6: Build Prompts
def build_prompt(item, lang):
    if lang == "en":
        return (
            f"Give me a complete summary of the following event, provide as much detail as possible: {item}"
        )
    elif lang == "es":
        return (
            f"Dame un resumen completo del siguiente evento, proporcionando la mayor cantidad de detalle posible.: {item}"
        )

In [ ]:
# Step 7: Query Model
SYSTEM_PROMPT_EN = (
    "You are a helpful assistant that provides factual information about "
    "historical events. Be factual and concise."
)
SYSTEM_PROMPT_ES = (
    "Eres un asistente útil que proporciona información factual sobre "
    "eventos históricos. Sé preciso y conciso."
)

def query_model(pipe, prompt, lang):
    system = SYSTEM_PROMPT_EN if lang == "en" else SYSTEM_PROMPT_ES
    messages = [
        {"role": "user", "content": f"{system}\n\n{prompt}"},
    ]
    output = pipe(messages, max_new_tokens=512, do_sample=False)
    return output[0]["generated_text"][-1]["content"]

In [ ]:
# Step 8: Run Experiment
MODEL_NAMES = [
    "google/gemma-2-2b-it",
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]

dataset = natural_disasters

existing = load_jsonl(MODEL_OUTPUTS_PATH)
if existing is not None:
    all_results = existing
    print("Skipping inference — loaded from saved file.")
else:
    all_results = []
    for model_name in MODEL_NAMES:
        print(f"\n{'='*60}")
        print(f"Loading: {model_name}")
        pipe = pipeline(
            "text-generation",
            model=model_name,
            torch_dtype=torch.float16,
            device_map="auto",
        )

        for item in dataset:
            print(f"  Prompting: {item}")
            output_en = query_model(pipe, build_prompt(item, "en"), "en")
            output_es = query_model(pipe, build_prompt(item, "es"), "es")
            all_results.append({
                "event_en":  item,
                "model":     model_name,
                "output_en": output_en,
                "output_es": output_es,
            })

        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Done with {model_name}")

    save_jsonl(all_results, MODEL_OUTPUTS_PATH)



Loading: google/gemma-2-2b-it


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

  Prompting: 2008 Sichuan earthquake


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Prompting: 2011 Christchurch earthquake


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done with Qwen/Qwen2.5-3B-Instruct
  Saved 20 records -> /content/output/natural_disasters/model_outputs.jsonl


In [ ]:
# Step 9: split results into models and language
gemma_es = [{"event_name": r["event_en"], "output_es":r["output_es"]} for r in all_results if r["model"] == "google/gemma-2-2b-it"]
gemma_en = [{"event_name": r["event_en"], "output_en":r["output_en"]} for r in all_results if r["model"] == "google/gemma-2-2b-it"]

qwen_es = [{"event_name": r["event_en"], "output_es":r["output_es"]} for r in all_results if r["model"] == "Qwen/Qwen2.5-3B-Instruct"]
qwen_en = [{"event_name": r["event_en"], "output_en":r["output_en"]} for r in all_results if r["model"] == "Qwen/Qwen2.5-3B-Instruct"]


In [ ]:
# Step 10: NER Tracking for model outputs
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
ENTITY_TYPES = ["person", "organization", "location", "country", "date", "event"]

MODEL_SHORT = {
    "google/gemma-2-2b-it":             "gemma",
    "Qwen/Qwen2.5-3B-Instruct":         "qwen",
    "meta-llama/Llama-3.2-3B-Instruct": "llama",
}


def ner_entities_chunked(text, entity_types, chunk_size=350, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += chunk_size - overlap
    seen = set()
    all_entities = []
    for chunk in chunks:
        for e in gliner_model.predict_entities(chunk, entity_types, threshold=0.5):
            key = (e["text"].lower().strip(), e["label"])
            if key not in seen:
                seen.add(key)
                all_entities.append((e["text"], e["label"]))
    return all_entities

# Split results by model
gemma_en = [r for r in all_results if r["model"] == "google/gemma-2-2b-it"]
gemma_es = [r for r in all_results if r["model"] == "google/gemma-2-2b-it"]
qwen_en  = [r for r in all_results if r["model"] == "Qwen/Qwen2.5-3B-Instruct"]
qwen_es  = [r for r in all_results if r["model"] == "Qwen/Qwen2.5-3B-Instruct"]
llama_en = [r for r in all_results if r["model"] == "meta-llama/Llama-3.2-3B-Instruct"]
llama_es = [r for r in all_results if r["model"] == "meta-llama/Llama-3.2-3B-Instruct"]

def get_or_compute_ner(records, lang, model_short):
    path = ner_path(model_short, lang)
    cached = load_entities(path)
    if cached is not None:
        return cached
    print(f"  Computing NER for {model_short} / {lang}...")
    key = f"output_{lang}"
    result = [ner_entities_chunked(r[key], ENTITY_TYPES) for r in records]
    save_entities(result, path)
    return result

gemma_en_entities = get_or_compute_ner(gemma_en, "en", "gemma")
gemma_es_entities = get_or_compute_ner(gemma_es, "es", "gemma")
qwen_en_entities  = get_or_compute_ner(qwen_en,  "en", "qwen")
qwen_es_entities  = get_or_compute_ner(qwen_es,  "es", "qwen")
llama_en_entities = get_or_compute_ner(llama_en, "en", "llama")
llama_es_entities = get_or_compute_ner(llama_es, "es", "llama")

print("NER extraction complete.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

  Computing NER for gemma / en...


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 499 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 451 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 432 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 472 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-p

  Saved -> /content/output/natural_disasters/ner/gemma_en.json
  Computing NER for gemma / es...


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 495 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 494 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 497 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 506 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-p

  Saved -> /content/output/natural_disasters/ner/gemma_es.json
  Computing NER for qwen / en...
  Saved -> /content/output/natural_disasters/ner/qwen_en.json
  Computing NER for qwen / es...
  Saved -> /content/output/natural_disasters/ner/qwen_es.json
NER extraction complete.


In [ ]:
# Step 11: Get Ground Truth from Wikipedia pages
from collections import Counter

def get_wikipedia_summary(title, lang='en'):
    wiki = wikipediaapi.Wikipedia(
        language=lang,
        user_agent="cross-lingual-llm-project (stevenacamachoperez@gmail.com)"
    )
    page = wiki.page(title)
    return page.summary if page.exists() else None

cached_gt = load_entities(GT_ENTITIES_PATH)
if cached_gt is not None:
    all_gt_entities = cached_gt
    print("Skipping Wikipedia fetch — loaded from saved file.")
else:
    all_summaries = [get_wikipedia_summary(item, "en") for item in dataset]
    all_gt_entities = []
    for i, summary in enumerate(all_summaries):
        gt_ents = ner_entities_chunked(summary, ENTITY_TYPES) if summary else []
        all_gt_entities.append(gt_ents)
        print(f"  {dataset[i]}: {len(gt_ents)} GT entities")
    save_entities(all_gt_entities, GT_ENTITIES_PATH)

# Save gt_entity_counts.csv — one row per event with total + per-type counts
gt_count_rows = []
for i, entities in enumerate(all_gt_entities):
    type_counts = Counter(lbl for _, lbl in entities)
    gt_count_rows.append({
        "event":          dataset[i],
        "total_entities": len(entities),
        **{t: type_counts.get(t, 0) for t in ENTITY_TYPES},
    })

df_gt_counts = pd.DataFrame(gt_count_rows)
df_gt_counts.to_csv(GT_ENTITY_COUNTS_PATH, index=False)
print(f"Saved GT entity counts -> {GT_ENTITY_COUNTS_PATH}")
print(df_gt_counts.to_string())

print("\nGround truth ready.")


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 401 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  2004 Indian Ocean earthquake and tsunami: 20 GT entities
  1976 Tangshan earthquake: 6 GT entities
  1920 Haiyuan earthquake: 12 GT entities


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 423 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  1923 Great Kantō earthquake: 22 GT entities
  1948 Ashgabat earthquake: 7 GT entities


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 446 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  2011 Tōhoku earthquake and tsunami: 21 GT entities
  1995 Great Hanshin earthquake: 8 GT entities


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 452 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  2023 Turkey–Syria earthquakes: 24 GT entities


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 443 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  2008 Sichuan earthquake: 15 GT entities
  2011 Christchurch earthquake: 14 GT entities
  Saved -> /content/output/natural_disasters/ground_truth/gt_entities.json
Saved GT entity counts -> /content/output/natural_disasters/ground_truth/gt_entity_counts.csv
   event  total_entities  person  organization  location  country  date
0      4              20       0             1         9        4     2
1      1               6       0             0         2        1     2
2      3              12       0             0         5        2     2
3      7              22       1             1        10        1     2
4      0               7       0             3         1        1     2
5      6              21       0             1        10        1     3
6      2               8       0             0         3        1     2
7      7              24       0             3         6        4     4
8      3              15       0             2         6        1     3
9      2              

In [ ]:

# Step 12: Normalize entities
import re

def normalize(entity):
    entity = entity.lower().strip()
    entity = re.sub(r"[^\w\s]", "", entity)
    return entity

In [ ]:
# Step 13: Compare raw entity counts to ground truth
from collections import Counter

model_count_rows = []
for label, entities_list in [
    ("gemma_en", gemma_en_entities),
    ("gemma_es", gemma_es_entities),
    ("qwen_en",  qwen_en_entities),
    ("qwen_es",  qwen_es_entities),
]:
    for i, entities in enumerate(entities_list):
        type_counts = Counter(lbl for _, lbl in entities)
        model_count_rows.append({
            "event":          dataset[i],
            "model_lang":     label,
            "total_entities": len(entities),
            **{t: type_counts.get(t, 0) for t in ENTITY_TYPES},
        })

df_model_counts = pd.DataFrame(model_count_rows)
df_model_counts.to_csv(MODEL_ENTITY_COUNTS_PATH, index=False)
print(f"Saved model entity counts -> {MODEL_ENTITY_COUNTS_PATH}")
print(df_model_counts.to_string())

def compute_ratios(entities_list, gt_entities_list, label):
    return [
        {
            "event":        dataset[i],
            "model_lang":   label,
            "model_count":  len(entities),
            "gt_count":     len(gt_entities_list[i]),
            "ratio":        round(len(entities) / len(gt_entities_list[i]), 4)
                            if len(gt_entities_list[i]) > 0 else None,
        }
        for i, entities in enumerate(entities_list)
    ]

ratio_rows = (
    compute_ratios(gemma_en_entities, all_gt_entities, "gemma_en") +
    compute_ratios(gemma_es_entities, all_gt_entities, "gemma_es") +
    compute_ratios(qwen_en_entities,  all_gt_entities, "qwen_en")  +
    compute_ratios(qwen_es_entities,  all_gt_entities, "qwen_es")
)

df_counts = pd.DataFrame(ratio_rows)
df_counts.to_csv(ENTITY_COUNTS_PATH, index=False)
print(f"Saved entity count ratios -> {ENTITY_COUNTS_PATH}")
print(df_counts.to_string())


In [ ]:
# Step 14: Compare entity type quantities across languages
from collections import Counter

def type_counts_rows(entities_list, label):
    rows = []
    for i, entities in enumerate(entities_list):
        counts = Counter(lbl for _, lbl in entities)
        rows.append({
            "event":      dataset[i],
            "model_lang": label,
            **{t: counts.get(t, 0) for t in ENTITY_TYPES},
        })
    return rows

type_rows = (
    type_counts_rows(gemma_en_entities, "gemma_en") +
    type_counts_rows(gemma_es_entities, "gemma_es") +
    type_counts_rows(qwen_en_entities,  "qwen_en")  +
    type_counts_rows(qwen_es_entities,  "qwen_es")
)

df_types = pd.DataFrame(type_rows)
df_types.to_csv(TYPE_COUNTS_PATH, index=False)
print(f"Saved entity type counts -> {TYPE_COUNTS_PATH}")
print(df_types.to_string())


In [ ]:
# Step 15: Compare entity types found to ground truth
def gt_type_ratio_rows(entities_list, gt_list, label):
    rows = []
    for i, entities in enumerate(entities_list):
        model_counts = Counter(lbl for _, lbl in entities)
        gt_counts    = Counter(lbl for _, lbl in gt_list[i])
        row = {"event": dataset[i], "model_lang": label}
        for t in ENTITY_TYPES:
            gt_val = gt_counts.get(t, 0)
            row[f"{t}_model"] = model_counts.get(t, 0)
            row[f"{t}_gt"]    = gt_val
            row[f"{t}_ratio"] = round(model_counts.get(t, 0) / gt_val, 4) if gt_val > 0 else None
        rows.append(row)
    return rows

gt_ratio_rows = (
    gt_type_ratio_rows(gemma_en_entities, all_gt_entities, "gemma_en") +
    gt_type_ratio_rows(gemma_es_entities, all_gt_entities, "gemma_es") +
    gt_type_ratio_rows(qwen_en_entities,  all_gt_entities, "qwen_en")  +
    gt_type_ratio_rows(qwen_es_entities,  all_gt_entities, "qwen_es")
)

df_gt_ratios = pd.DataFrame(gt_ratio_rows)
df_gt_ratios.to_csv(GT_RATIOS_PATH, index=False)
print(f"Saved GT type ratios -> {GT_RATIOS_PATH}")
print(df_gt_ratios.to_string())


Saved GT type ratios -> /content/output/natural_disasters/analysis/gt_type_ratios.csv
                                       event model_lang  person_model  person_gt  person_ratio  organization_model  organization_gt  organization_ratio  location_model  location_gt  location_ratio  country_model  country_gt  country_ratio  date_model  date_gt  date_ratio  event_model  event_gt  event_ratio
0   2004 Indian Ocean earthquake and tsunami   gemma_en             0          0           NaN                   1                1              1.0000               3            9          0.3333              9           4           2.25           1        2      0.5000            1         4       0.2500
1                   1976 Tangshan earthquake   gemma_en             0          0           NaN                   3                0                 NaN               2            2          1.0000              1           1           1.00           1        2      0.5000            1         1    